## Pybridizer basic usage pipeline

Follow the steps in this notebook to generate HCR probes.

### Before you start

Check at the top-right to ensure this notebook is using the correct python environment. It should say `pybridize` (or similar) next to the empty circle. 

If it says something else (e.g. `Python 3 (ipykernel)`), change it by selecting `Kernel > Change kernel > pybridize` in the menu at the top. 

If the `pybridize` option is not available, you must install the environment first (and activate it as an ipykernel); check the README for more info. If you don't have any experience with python packages, you may want to work with someone who does.

### Prepare your target sequence

You will need the mRNA sequence of your target gene in FASTA file format.

- Go to the [NCBI](https://www.ncbi.nlm.nih.gov/) webpage
- Type the gene and species into the search bar, e.g. `sox10 danio rerio`
- It should find a `GENE` entry for your gene; click on `RefSeq products`
- Select one of the entries in the `Transcript accession` column
    - Usually the top one is the correct one
    - If you're unsure, check that the protein size (`Length (aa)` column) is right
    - If there are still multiple options, go to the genome browser and figure out which is best
- Clicking on the transcript accession number should get you to a GenBank file with a sensible header, e.g.:
    - `Danio rerio SRY-box transcription factor 10 (sox10), mRNA`
- Click the little `FASTA` link under the header to get the FASTA format
- At the top right, click `Send to:`, then select `File` as destination, ensure the format is `FASTA`, and click `Create File`
- Save the file with a clear name, e.g. `sox10-drerio-NM_131875.1.fasta` (`[gene]_[species]_[accession].fasta`)

### Getting started

If you are reusing this notebook, first clear out any results left from previous runs; in the menu at the top, click `Kernel > Restart & Clear Output` and then confirm by pressing the red button.

Now proceed to execute each of the code cells below, step by step. You can do so by selecting the cell and pressing `Shift+Enter` or by clicking the `> Run` button at the top.

Import required packages and the functions from the pybridizer probe design module:

In [ ]:
import os

from pybridizer import (
    read_fasta, create_oligos, blast_oligos, fetch_geneIDs,
    filter_and_rank, add_hairpin, plot_sequence_alignment,
)

Set the input and output folders. By default, these point to the folders in this repository, so if using the default make sure you move the fasta file of your target sequence (downloaded above) into the `input` folder.

You can also provide other paths by copy-pasting the full path (e.g. starting with `C:\` on Windows) between the scare quotes.

In [ ]:
input_dir = r"../input"
output_dir = r"../output"

### Construct candidate oligos

First, the transcript sequence for your target gene must be loaded from the FASTA file you prepared above.

Copy the full file name (including `.fasta` file ending) into the `file_name` variable, then run the cell.

It should print the correct ID, header, and length.

In [ ]:
file_name = r"sox10-drerio-NM_131875.1.fasta"

ID, desc, sequence = read_fasta(os.path.join(input_dir, file_name))

Now, the transcript sequence needs to be tiled into candidate oligos (default:25 base length, 2 base gap).

Optional: You can also adjust the length of the oligo, gap between oligos and frame start position (0 i.e. beginning of the transcript by default). This is for advanced use cases only.

In [ ]:
oligos_all = create_oligos(sequence, oligo_length=25, gap_length=2, frame_start_position=0)

### Filter out the best oligo pairs

First, a batch BLAST search is performed for each of the oligos against the local database. This is to find all potential off-target binding sites for the oligos.

<font color=crimson>**Warning:**</font> BLAST+ must be installed and a BLAST database for the relevant organism must have been built using `makeblastdb` on genomic mRNA data downloaded from NCBI. This is a pre-requisite for the BLAST function to work. See  the README for instructions. This can be complicated, so if it is not already done and you don't have experience with command line applications, you may want to work with someone who does.

[Windows only:] Running the cell below will show you which databases are available.

In [ ]:
!blastdbcmd -list %BLASTDB%

_You must select the correct species for your target sequence!_

For example, if the above outputs this...

```
C:\Users\yourname\NCBI\blast-2.17.0+\db\zebrafish_transcripts Nucleotide
C:\Users\yourname\NCBI\blast-2.17.0+\db\xenopus_transcripts Nucleotide
```

...and you are designing probes for a zebrafish gene, use `zebrafish_transcripts` as the dbname and run the cell below.

In [ ]:
base_name = file_name.split(".fasta")[0]
faa_fpath = os.path.join(output_dir, base_name + "_oligos.faa")
result_fpath = os.path.join(output_dir, base_name + "_result_tab.txt")

blast_result = blast_oligos(
    oligos_all, 
    dbname="zebrafish_transcripts",  # Choose correct database here!
    faa_fpath=faa_fpath, 
    result_fpath=result_fpath,
)

The resulting dataframe lists the query results for each oligo, including hits (off-target matches with other transcripts in the database), length of the matches, match positions, and match scores. The first 5 rows of the dataframe are shown below. You don't need to do anything with this unless you are an advanced user with special requirements for your probe design. The results are also saved into output files.

In [ ]:
blast_result.head()

Some of the hits will be mRNA database entries for the same gene, which are redundant. In the next step, the gene IDs for each of the hits is fetched, which can be used to identify these cases.

You must choose the correct Taxonomy ID ("taxid"), e.g. `7955` for *Danio rerio*. Search for your organism [here](https://www.ncbi.nlm.nih.gov/taxonomy) to find it.

Note that this step requires an active internet connection. Also, it is optional; to skip it, comment out the top row and comment in the bottom row, then run the cell.

In [ ]:
blast_result_genes = fetch_geneIDs(blast_result, taxid='7955')  # Comment out to skip this step
#blast_result_genes = blast_result  # Comment in to skip this step

View the blast results with gene IDs included:

In [ ]:
blast_result_genes.head()

Next, the oligos need to be filtered based on three criteria:

- The GC content (default range `37-85%`)
- The Melting Temperature (default range `47-85°C`)
- Any neighboring oligos that both hit the same off-target transcript (this ensures target specificity of the probes!)

The remaining neighboring oligo pairs are ranked according to specificity.

As a default, we hope to get 12 or more suitable probe pairs, in which case we pick the top 12 (as this fits into the rows of a 96-well plate). 8-11 probe pairs should also work fine, and we've had success with as few as 6. For low-abundance genes, a larger number of pairs (say 16-20) could be used to maximize sensitivity, assuming of course a sufficient number of pairs is identified here.

Advanced users: If you get too few probes, you can try widening the GC and/or MT ranges. However, you may want to read up first (or ask some super-expert) regarding the most sensible potential adjustments...

In [ ]:
probe_datasheet = filter_and_rank(
    oligos_all, blast_result_genes,
    Tm_range=[47,85],
    GC_range=[0.37,0.85]
)

View the probe datasheet for the optimal set of HCR oligo pairs after filtering.

In [ ]:
probe_datasheet

### Add the hairpin-binding sequence

**You must choose the correct B-number for your hairpins:**

By convention, we use the following hairpins in the lab (but of course other options are possible):

- **B1**: `Alexa Fluor 488` (green)
- **B3**: `Alexa Fluor 546` (red)
- **B5**: `Alexa Fluor 647` (far-red)

Choose the correct B-number and run the next cell

In [ ]:
#probe_Bnum = "B1"  # usually green
#probe_Bnum = "B3"  # usually red
probe_Bnum = "B5"  # usually far-red

hcr_probes = add_hairpin(probe_datasheet, probe_Bnum)

### View and export the final HCR probes

View the final HCR probes for the chosen hairpin.

The `hcr_probes` datasheet has the following columns:

- `Oligo1_position` & `Oligo2_position`: Location of the oligo along the transcript

* `Oligo1_Sequence` & `Oligo2_Sequence`: Oligo pairs (reverse complements of sequence on transcript)

- `Score (avg hits)`: How many off-target transcripts on average could partially bind to either of the two oligos
    - A lower score means higher specificity of the probe pair
    - Note that all pairs where both probes bind on the same off-target mRNA have already been filtered out
    - Therefore, a high off-target score here does *not* mean that there will be off-target signal in the staining!
    - Instead, the only issue may be that individual probes get sequestered away to their off-target sites, leading to reduced signal
    - This can potentially be countered by using a higher concentration for probes that have a high score here

* `HCRprobe1` and `HCRprobe2` are the final hairpin-appended oligos that should be ordered as ssDNA
    - As discussed above, we usually order the top 12 pairs (i.e. with the lowest Score)
    - You may order fewer (it can work with as few as 6) if there are not enough
    - You can order more to get better signal for low-abundance genes

In [ ]:
hcr_probes

Next, we can visualize the alignment of the oligo pairs to the target transcript sequence.

You can select the top **n** probe pairs to include in the figure/alignment by setting the `top_n` parameter. Set it to `None` to show all probe pairs.

This plot (as well as a FASTA file containing the alignment, which can be loaded e.g. in SnapGene) are saved into the output folder.

In [ ]:
top_n = 12  # Include top 12 probe pairs
#top_n = None  # Include all probe pairs

align_fig_fpath = os.path.join(output_dir, base_name + "_oligo-alignment.png")
align_fasta_fpath = os.path.join(output_dir, base_name + "_oligo-alignment.fasta")

fig, records = plot_sequence_alignment(
    sequence, ID, probe_datasheet, 
    top_n=top_n, figsize=(16, 8), 
    save_path=align_fig_fpath, 
    output_fasta=align_fasta_fpath,
)

Finally, the designed HCR probes datasheet can be saved as an Excel file.

In [ ]:
excel_fpath = os.path.join(output_dir, base_name + f"_probes-{probe_Bnum}.xlsx")

hcr_probes.to_excel(excel_fpath)

### Order your probes

The sequences to order are in the last two columns of the excel sheet (`HCRprobe1` and `HCRprobe2`).

Pick the top n probe pairs (again, usually the top 12) and order them. We used to order from Sigma with below configuration. Nowadays, you may alternatively order from IDT with similar parameters.

- Format: In solution (Water)
- Purification: Desalt
- Concentration (uM): 100
- Fixed Volume Per Well: No
- Scale: 0.025umole
- Plate type: Shallow Well (200uL)
- Shipping Options: Ambient (Room Temperature)

When ordering probes for multiple genes, it is strongly recommended to order them in 96-well plates (two rows per probe pair) and with "normalized concentration" (meaning they arrive at 100uM concentration). Otherwise, you'll have to deal with dozens of tubes and add a custom amount of water to each of them to get the right final concentrations...

(As of 2026, the price for 12 probe pairs, i.e. 24 sequences, comes to about GBP 120. Even at the smallest synthesis scale, the amount of probe you get is enough for thousands of samples. Thus, the main factor to consider in terms of cost per experiment are the hairpins, not the probes.)

Upon receiving the probes, create a probe mix by combining 10ul of each 100uM probe and filling up to 1ml. If you order 12 probe pairs, that's 24 individual oligos, so the combination of 10ul each will amount to 240ul, meaning you'll have to add 760ul of nuclease-free water to get 1ml of 1uM probe mix.

The standard HCR protocol then uses 2ul of this mix (=2pmol) in 500ul of Probe Hybridization Buffer.

You can store both the individual probes and the probe mix at -20°C for a very long time. 